# Task 2 and Task 3 Analysis Evidence - dabi0142


## Member Scope


In [ ]:
from dataclasses import replace

import pandas as pd

from data2001.config import load_settings
from data2001.db.engine import create_engine_from_settings
from data2001.pipeline import execute_workflow_steps
from data2001.task4.charts import (
    build_bottom_sa2_bar,
    build_poi_group_distribution,
    build_score_histogram,
    build_score_income_scatter,
    build_top_sa2_bar,
)
from data2001.task4.tables import build_sa4_summary_table, build_top_bottom_table
from data2001.task4.maps import build_score_choropleth_map, build_poi_density_choropleth_map, build_poi_point_scatter_map

from data2001.task4.queries import (
    load_api_extraction_summary,
    load_correlation_summary,
    load_poi_group_counts,
    load_poi_points,
    load_sa2_scores,
    load_score_income,
    load_score_input_summary,
    load_spatial_join_summary,
)


MEMBER_UNIKEY = "dabi0142"

base_settings = load_settings("configs/local.yaml")
member_sa4 = (base_settings.task2_import.selected_sa4_by_member.get(MEMBER_UNIKEY) or "").strip()
if not member_sa4:
    raise ValueError(f"No SA4 configured for {MEMBER_UNIKEY}. Fill configs/local.yaml selected_sa4_by_member.")

settings = replace(
    base_settings,
    task2_import=replace(
        base_settings.task2_import,
        crawl_scope="selected_sa4",
        selected_sa4_by_member={MEMBER_UNIKEY: member_sa4},
    ),
    task3_score=replace(base_settings.task3_score, score_universe="selected_sa4"),
)
engine = create_engine_from_settings(settings.database)

member_scope = pd.DataFrame([{"unikey": MEMBER_UNIKEY, "selected_sa4": member_sa4}])
display(member_scope)

## Single-SA4 Full Workflow Run


In [ ]:
workflow_steps = [
    "init_db",
    "clear_db",
    "import_boundaries",
    "import_poi",
    "import_income",
    "compute_score",
]

workflow_summary = execute_workflow_steps(
    engine,
    settings,
    workflow_steps,
    title=f"{MEMBER_UNIKEY} single-SA4 full rebuild",
)
display(workflow_summary)

## Single-SA4 Database Verification


In [ ]:
schema = settings.database.schema_name

display(pd.read_sql(
    f"""
    SELECT sa4_name, COUNT(*) AS sa2_count
    FROM {schema}.sa2
    GROUP BY sa4_name
    """,
    engine,
))

## Task 2 Evidence: API Extraction and Spatial Join


In [ ]:
display(load_api_extraction_summary(settings))
display(load_spatial_join_summary(engine, settings))

## Task 3 Evidence: Score Calculation


In [ ]:
display(load_score_input_summary(engine, settings))

scores = load_sa2_scores(engine, settings)
score_map_areas = load_sa2_scores(engine, settings, include_excluded=True)
display(scores.head())
display(build_top_bottom_table(scores, n=settings.charts.top_n))

## Individual Visual Analysis


In [ ]:
poi_groups = load_poi_group_counts(engine, settings)
poi_points = load_poi_points(engine, settings, limit=settings.dashboard.poi_limit)
score_income = load_score_income(engine, settings)

### Score Distribution


In [ ]:
build_score_histogram(scores, nbins=settings.charts.score_histogram_nbins).show()

### Top and Bottom SA2 Scores


In [ ]:
build_top_sa2_bar(scores, n=settings.charts.top_n).show()
build_bottom_sa2_bar(scores, n=settings.charts.top_n).show()

### Score Choropleth Map


In [ ]:
build_score_choropleth_map(score_map_areas).show()

### Population-Adjusted POI Density Map


In [ ]:
build_poi_density_choropleth_map(score_map_areas).show()


### POI Point Map


In [ ]:
build_poi_point_scatter_map(poi_points).show()

### POI Group Distribution


In [ ]:
build_poi_group_distribution(poi_groups).show()

### Score and Median Income


In [ ]:
build_score_income_scatter(score_income).show()

## Correlation and Interpretation Notes


In [ ]:
display(load_correlation_summary(engine, settings))